### 양상블
- 여러개의 분류모델을 조합해서 도나은 성능을 내는 방법
- Decision Tree 모델을 증가시켜 나온 랜덤포레스트가 대표적임

#### 랜덤 포레스트
##### 랜덤포레스트 훈련방법
- 부트스트랩 샘플: 중복을 허용하는 샘플링 방법
- 샘플링 후에 샘플을 복구하고 다시 샘플링하는 방법
- 이와 같이 진행하는 이유는 결정트리에서 과대적합을 방지할 수 있기 때문이다.
- 각 결정트리에서 나오는 확률의 합을 트리갯수로 나누어서 결정짓는 모델
- 부트스트랩의 샘플링 갯수: 특성의 갯수의 제곱근

In [2]:
import pandas as pd

In [3]:
wine = pd.read_csv('data/wine.csv')

In [4]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [6]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = \
    train_test_split(
        data,
        target,
        test_size = 0.2,
        random_state = 42,
        stratify = target
    )

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

In [9]:
rf = RandomForestClassifier(
    n_jobs = -1 # 
)
scores = cross_validate(
    rf,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs = -1
)
scores

{'fit_time': array([0.11916804, 0.11336303, 0.11066604, 0.1173892 , 0.11684799]),
 'score_time': array([0.01411009, 0.0146451 , 0.014539  , 0.01402807, 0.01512599]),
 'test_score': array([0.90288462, 0.90096154, 0.88546679, 0.88642926, 0.87584216]),
 'train_score': array([0.99783498, 0.99783498, 0.9983165 , 0.998076  , 0.998076  ])}

In [12]:
print('Train: ', scores['train_score'].mean())
print('Valid: ', scores['test_score'].mean())

Train:  0.9980276897524913
Valid:  0.8903168727326571


In [13]:
rf.fit(train_input, train_target)
rf.score(test_input, test_target)

0.8930769230769231

In [14]:
print(rf.feature_importances_)

[0.23675046 0.49910045 0.2641491 ]


#### Extra Tree
- 기본적으로 100개의 트리를 사용
- 노드 분할시 특성의 제곱근의 갯수를 사용
- 특성의 선택을 랜덤으로 한다.
- 특성 선택을 랜덤하게 하므로 속도는 랜덤포레스트보다 빠르다.

In [17]:
from sklearn.ensemble import ExtraTreesClassifier

In [21]:
et = ExtraTreesClassifier(
    n_jobs = -1 # 
)
scores = cross_validate(
    et,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs = -1
)
scores
print('Train: ', scores['train_score'].mean())
print('Valid: ', scores['test_score'].mean())

Train:  0.9981720130385032
Valid:  0.8905101058710297


#### Gradient Boosting
- 가장 유명한 알고리즘중 하나이다.
- 경사하강법처럼 손실함수 사용
- 손실함수를 보고 트리를 추가하여 최적의 값을 도출
- max_deprh는 3으로 제어됨, 과대적합 방지
- 단점은 손실함수를 확인하고 트리는 추가하는 모델임으로 n_jobs(병렬처리)를 할 수 없다.

In [19]:
from sklearn.ensemble import GradientBoostingClassifier

In [22]:
gb = GradientBoostingClassifier()
scores = cross_validate(
    gb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs = -1
)
scores
print('Train: ', scores['train_score'].mean())
print('Valid: ', scores['test_score'].mean())

Train:  0.886136193834053
Valid:  0.8714622047827053


#### Histogram Gradient Boosting
- 훈련데이터를 256개의 구간으로 나누어서 훈련시키는 방법
- 특성의 범위가 제한되어 있어 빠른 속도를 제공한다.
- 제한된 구간임으로 과대 적합을 방지한다.

In [23]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [25]:
hgb = HistGradientBoostingClassifier()
scores = cross_validate(
    hgb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs = -1
)
scores
print('Train: ', scores['train_score'].mean())
print('Valid: ', scores['test_score'].mean())

Train:  0.930536541746549
Valid:  0.8780021470348707


#### LightGBM
- Gradient Boosting에서 출발
- 

In [27]:
# !pip install lightgbm

In [30]:
from lightgbm import LGBMClassifier

In [31]:
lgb = LGBMClassifier()
scores = cross_validate(
    lgb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs = -1
)
scores
print('Train: ', scores['train_score'].mean())
print('Valid: ', scores['test_score'].mean())

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train:  0.934577605325741
Valid:  0.8805047382838529


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/